# V2 known-good validation before webcam demo

This notebook uses the exact XSA/model set from the Vitis project reported as **91.5% accuracy**.

Goal: first prove that the Jupyter host path reproduces correct predictions on the *embedded CIFAR-10 test images*. Only after that should we judge webcam results.

In [ ]:
import time
import numpy as np
import cv2
from IPython.display import display, Image, clear_output
from pynq import Overlay

from npu_v2_jupyter_known_good import NpuV2, center_crop_resize_rgb

BIT = "v2_known_good.bit"
MODEL = "v2_known_good_full.npz"


## 1. Program the exact known-good bitstream and preload the exact model

In [ ]:
ol = Overlay(BIT, download=True)
print(list(ol.ip_dict.keys()))

npu = NpuV2(overlay=ol, ip_name="myip_0")
print("NPU base = 0x%08X" % npu.base_addr)

npu.load_model_npz(MODEL)
npu.preload_model(progress=True)


## 2. Validate on embedded CIFAR test images

Do **not** start with image 0 only: CIFAR-10 test image 0 has label `cat`. The first 20 labels include ship, airplane, frog, automobile, etc., so a cat-collapse is immediately obvious.

In [ ]:
result20 = npu.validate_embedded_testset(
    count=20,
    start=0,
    channel_guard_s=0.001,
    progress=True
)
print("\n20-image accuracy: %d/%d = %.1f%%" %
      (result20["correct"], result20["count"], result20["accuracy"]*100))
print("Average Jupyter host-path time: %.2f ms" % result20["avg_ms"])


### Interpretation

- If the 20-image result is roughly correct and predictions vary by class, the PL/model/Jupyter driver are functioning; webcam misclassification is then mostly input-domain/generalization.
- If most/all embedded CIFAR images still predict `cat`, the problem is in the Python MMIO/preload/channel-handshake path, not the trained model.
- If this 20-image check looks good, run the full 1000-image check below. It will be much slower than Vitis because Python performs thousands of AXI-Lite MMIO writes per image.

In [ ]:
# Optional full reproduction test. This may take a long time in Python.
# result1000 = npu.validate_embedded_testset(
#     count=1000, start=0, channel_guard_s=0.001, progress=False
# )
# print("Accuracy: %d/%d = %.2f%%" %
#       (result1000["correct"], result1000["count"], result1000["accuracy"]*100))


## 3. Only after CIFAR validation: webcam one-frame test

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
assert cap.isOpened()
ok, frame = cap.read()
cap.release()
assert ok

rgb32 = center_crop_resize_rgb(frame, cv2)
pred, scores, q, inv_q, dt = npu.infer_rgb32(rgb32)

print("Prediction:", npu.classes[pred])
print("param 530:", inv_q)
print("host path: %.2f ms" % (dt*1000))
print("scores:", scores.tolist())

preview = cv2.resize(cv2.cvtColor(rgb32, cv2.COLOR_RGB2BGR),
                     (320,320), interpolation=cv2.INTER_NEAREST)
ok, buf = cv2.imencode(".jpg", preview)
display(Image(data=buf.tobytes()))
